# FallDeFi baseline on the shared split

Runs the FallDeFi adaptation (`analysis/baselines/falldefi/`, see [the guide](../../docs/baselines/falldefi.md)): STFT spectrograms, power-burst events and an RBF SVM on event features, with one trace per action cue widened by 1 s.

Every approach is scored on the attempts that one `03_training` run held out
(`window_split.parquet`). The baselines train on that run's training and
validation attempts, because they choose their settings on training data only,
and are tested on its test attempts. Attempts that the baseline loads but the
split does not contain are left out of both sides and counted. Fall and
non-fall labels are read from the split, so the label mapping is the sequence
export's.

**Scope.** With `SAMPLE = True` in `02_preprocess` and `03_training`, the
split covers one session (Akmal, 9 November 13:15) and 2 s windows. Results on
it are a smoke check of the pipeline, not evidence of detection performance:
the session is small, informed the baseline adaptations, and comes from one
participant.

## Setup

Set `SPLIT_RUN` to a `03_training` run directory to pin the split; `None` uses the newest default-representation run that has one. Set `HELD_OUT_SESSION` to use the newest run that held out that participant session with the same `KEEP_SPLIT`.

In [ ]:
import json
import sys
from datetime import UTC, datetime
from pathlib import Path

import polars as pl
from IPython.display import display

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/fall_detection").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fall_detection.analysis._cache_keys import file_hash
from fall_detection.analysis.baselines.falldefi.dataset import load_traces
from fall_detection.analysis.baselines.falldefi.experiment import (
    run_experiment,
    save_experiment,
)
from fall_detection.analysis.baselines.falldefi.model import FallDeFiConfig
from fall_detection.analysis.comparison import (
    latest_training_run,
    split_assignment,
    split_labels,
    split_sessions,
)

# The held-out attempts come from a 03_training run; None selects the newest one.
SPLIT_RUN = None
# With SPLIT_RUN = None, the newest run holding out this session (None: random split).
HELD_OUT_SESSION = None
# With a holdout, datasets that kept their random split in 03_training.
KEEP_SPLIT = ("continuous_non_fall",)
TRAINING_RUNS = PROJECT_ROOT / "artifacts/notebooks/03_training"
split_run = SPLIT_RUN or latest_training_run(
    TRAINING_RUNS,
    "window_split.parquet",
    held_out_session=HELD_OUT_SESSION,
    keep_split=KEEP_SPLIT,
)
split_path = split_run / "window_split.parquet"
split = pl.read_parquet(split_path)
training_manifest = json.loads((split_run / "manifest.json").read_text())
ASSIGNMENT = split_assignment(split)
FALL_LABELS, NON_FALL_LABELS = split_labels(split)
SOURCES = [
    PROJECT_ROOT / "data/raw/sessions" / f"{session}.zip"
    for session in split_sessions(split)
]
CONFIG = FallDeFiConfig(random_seed=42)
RUN = (
    PROJECT_ROOT
    / "artifacts/notebooks/05_falldefi"
    / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
)
print(
    "Split:",
    split_run.relative_to(PROJECT_ROOT),
    "sample:",
    training_manifest.get("sample"),
)
print("Sessions:", [path.name for path in SOURCES])
print("Fall labels:", FALL_LABELS)
print("Non-fall labels:", NON_FALL_LABELS)

## Traces

The table counts traces and attempts per split and target; `unassigned` attempts are left out.

In [ ]:
traces = load_traces(
    SOURCES,
    root=PROJECT_ROOT,
    fall_labels=FALL_LABELS,
    non_fall_labels=NON_FALL_LABELS,
)
assigned = pl.DataFrame(
    {"group": traces.groups.tolist(), "target": traces.targets.tolist()}
).with_columns(
    pl.col("group").replace_strict(ASSIGNMENT, default="unassigned").alias("split")
)
display(
    assigned.group_by("split", "target")
    .agg(pl.len().alias("traces"), pl.col("group").n_unique().alias("attempts"))
    .sort("split", "target")
)

## Train and test

The threshold and the SVM are fitted on training traces only. Outputs go to `artifacts/notebooks/05_falldefi/<run>/svm/`.

In [ ]:
RUN.mkdir(parents=True, exist_ok=False)
result = run_experiment(traces, CONFIG, assignment=ASSIGNMENT)
save_experiment(result, RUN / "svm")
report = result.report
(RUN / "manifest.json").write_text(
    json.dumps(
        {
            "notebook": "05_falldefi",
            "split_run": str(split_run.relative_to(PROJECT_ROOT)),
            "split_sha256": file_hash(split_path),
            "training_manifest": training_manifest,
            "sources": [str(path.relative_to(PROJECT_ROOT)) for path in SOURCES],
            "fall_labels": list(FALL_LABELS),
            "non_fall_labels": list(NON_FALL_LABELS),
        },
        indent=2,
    )
    + "\n"
)
print("Results:", RUN.relative_to(PROJECT_ROOT))

## Held-out results

A trace is a predicted fall when any of its events is classified as one. Event detection and false alarms are reported as the experiment computes them.

In [ ]:
metrics = report["held_out"]
display(
    pl.DataFrame(
        [
            {
                "level": "trace",
                **{
                    name: metrics[name]
                    for name in (
                        "balanced_accuracy",
                        "precision",
                        "recall",
                        "f1",
                        "false_positive_rate",
                    )
                },
                "traces": sum(map(sum, metrics["confusion_matrix"])),
            }
        ]
    )
)
print("Event detection:", report["held_out_event_detection"])
print("False alarms:", report["held_out_false_alarms"])
print(
    "Test attempts:",
    len(report["split"]["test_attempts"]),
    "unassigned attempts left out:",
    len(report["split"]["unassigned_attempts"]),
)